In [0]:
# #Broadcast join vs sort-merge join decision

# %python
# from pyspark.sql.functions import broadcast

# # Rule: broadcast if one side < spark.sql.autoBroadcastJoinThreshold (default 10MB)
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "50MB")

# # Explicit broadcast hint
# result = spark.table("practice_db.orders").join(
#     broadcast(spark.table("practice_db.products")),
#     "product_id"
# )
# result.explain("formatted")  # look for BroadcastHashJoin in plan

# # Check actual join strategy used
# result.explain(True)

In [0]:
# #Custom partitioning for query optimization

# %python
# # Partition orders by year-month for time-range queries
# spark.table("practice_db.orders") \
#     .withColumn("year_month", date_format("order_date", "yyyy-MM")) \
#     .write \
#     .mode("overwrite") \
#     .partitionBy("year_month") \
#     .saveAsTable("practice_db.orders_partitioned")

# # Verify partition pruning is happening
# spark.sql("""
#   SELECT count(*) FROM practice_db.orders_partitioned 
#   WHERE year_month = '2023-06'
# """).explain()  # should show PartitionFilters in plan

In [0]:
# #Structured Streaming simulation

# %python
# # Simulate streaming from Delta table (common in interviews)
# from pyspark.sql.functions import window, count, sum as _sum

# stream_df = spark.readStream \
#     .format("delta") \
#     .table("practice_db.events")

# # Windowed aggregation — tumbling window
# windowed = stream_df \
#     .withWatermark("event_ts", "10 minutes") \
#     .groupBy(
#         window("event_ts", "1 hour"),
#         "event_type"
#     ).agg(
#         count("*").alias("event_count"),
#         countDistinct("user_id").alias("unique_users")
#     )

# # Write to memory sink for testing
# query = windowed.writeStream \
#     .outputMode("update") \
#     .format("memory") \
#     .queryName("event_counts") \
#     .trigger(processingTime="30 seconds") \
#     .start()

# # Query results
# spark.sql("SELECT * FROM event_counts ORDER BY window").show()

In [0]:
# #Great Expectations — data quality (trending in 2025 interviews)

# %python
# # pip install great_expectations
# import great_expectations as gx

# context = gx.get_context()

# # Define expectations on orders table
# validator = context.sources.pandas_api.read_dataframe(
#     spark.table("practice_db.orders").limit(10000).toPandas()
# )

# validator.expect_column_values_to_not_be_null("order_id")
# validator.expect_column_values_to_be_between("amount", min_value=0, max_value=10000)
# validator.expect_column_values_to_be_in_set("status", ["COMPLETED","CANCELLED","RETURNED"])
# validator.expect_column_values_to_be_unique("order_id")

# results = validator.validate()
# print(results.success)  # True/False
# print(results.statistics)  # % passing expectations